# Spatial Polarization

This notebook illustrates the `S` spatial polarization index implemented in `inequality.polarization`, based on the graph-intersection framework proposed in:

> Rey, S. J. (2026). Mind the gap and the map: measuring distributional and spatial income polarisation. *Spatial Economic Analysis*. https://doi.org/10.1080/17421772.2025.2606783

Polarization is distinct from inequality: following Esteban and Ray (1994), it combines *alienation* (how far apart groups are) with *group identification* (how homogeneous each group is). Zhang and Kanbur (2001) measure spatial polarization as a ratio of between-region to within-region inequality, but require an exogenously fixed partition of space into regions. Rey (2026) instead defines spatial polarization directly from two graphs built over the same set of `n` locations:

- an **attribute graph** `A`, connecting pairs of locations that fall in the same value class (e.g., both above or both below the median of some variable), and
- a **spatial graph** `G`, connecting pairs of locations that are geographic neighbours.

Intersecting `A` and `G` gives a graph `I` whose edges are pairs that are *both* attribute-similar and spatially adjacent. Comparing the number of connected components of `I`, `k_I`, against `k = max(k_A, k_G)` and `n` gives the index:

`S(A, G) = 1 - (k_I - k) / (n - k)`

`S` ranges from 0 (no overlap between spatial and attribute structure) to 1 (complete overlap: the connected pieces of the spatial graph line up exactly with the value classes). Splitting the attribute at the median (`k=2`) gives the *spatial bipolarization* index; using more than two classes generalizes to *spatial multipolarization*.

The `S` class below implements this index and adds a permutation-based inference procedure (randomly reshuffling the attribute values across locations) to obtain a pseudo p-value for the observed statistic - an extension beyond the purely descriptive index presented in the paper, which flags inference as a direction for future work.


In [ ]:
from inequality.polarization import S

In [ ]:
import libpysal
import pandas as pd
import numpy as np
from libpysal.weights import lat2W
from libpysal.graph import Graph

## A synthetic check: a regular 40x40 lattice

As a first, controlled example, we place `n = 1600` observations on a regular 40x40 lattice (contiguity via `lat2W`) and assign them the values `0, 1, ..., 1599` in row-major order. Because values increase steadily along each row before dropping back down at the start of the next, splitting at the median (`k=2`, the default) produces two attribute classes - low and high - that are each spatially contiguous within most rows. We should therefore expect a fairly high, though not perfect, spatial polarization value.

In [ ]:
y = np.arange(1600)

In [ ]:
df = pd.DataFrame({'y': y}, index=y)

In [ ]:
g = Graph.from_W(lat2W(40, 40))

By default `S` runs `permutations=999` random reshuffles of the attribute values to build a null distribution, so `p_value` reports how (un)likely the observed spatial polarization would be if the values were randomly scattered across locations instead of following their actual spatial arrangement. `n_jobs=4` parallelizes the permutations across four worker processes.

In [ ]:
res = S(df, g, 'y', n_jobs=4)

In [ ]:
res

Setting `permutations=0` skips the inference step entirely and returns just the observed statistic - useful when only the descriptive index is needed, or when many variables are being screened before running significance tests on the interesting ones.

In [ ]:
res = S(df, g, 'y', permutations=0)

In [ ]:
res

In [ ]:
res

The `labels` attribute exposes the underlying classification for every location: `a_labels` is the attribute (value-class) membership, `g_labels` the spatial-graph component membership, and `i_labels` the component membership in the intersection graph `I` used to compute `S`.

In [ ]:
res.labels

Since permutation inference is embarrassingly parallel, it's worth checking how much `n_jobs` helps in practice on this 1,600-observation lattice.

In [ ]:
%%timeit
res = S(df, g, 'y', n_jobs=4)

For reference, the full source of `S` - the union-find based component counting, quantile binning, and permutation loop - can be inspected directly.

In [ ]:
S??

## Case study: Mexican state income, 1940-2000

The paper's illustrative application - reproduced here - examines per-capita GDP for Mexico's 32 states (31 states plus Ciudad de Mexico) at ten-year intervals from 1940 to 2000. The series extends Esquivel's (1999) 1940-1990 data with the 2000 figures from Rey and Sastre-Gutierrez (2010), and ships with `libpysal.examples` as the `"mexico"` dataset.

Across this 70-year span the distribution of state incomes shows persistent right skew and a pronounced north-south divide, with low-income states concentrated in the south. The question the spatial polarization index is built to answer is not just *how unequal* incomes are (that's the job of Theil's T or the Gini index), but *how spatially organized* that inequality is - do similarly poor, or similarly rich, states cluster together geographically?

In [ ]:
import libpysal

In [ ]:
libpysal.examples.explain('mexico')

In [ ]:
mexico = libpysal.examples.load_example('mexico')

In [ ]:
mexico.get_file_list()

In [ ]:
import geopandas as gpd

In [ ]:
gdf = gpd.read_file(libpysal.examples.get_path('mexicojoin.shp'))

In [ ]:
gdf.plot()

In [ ]:
gdf.head()

In [ ]:
S?

The spatial graph `G` is built from queen contiguity among the 32 states (two states are neighbours if they share at least a boundary point). In the paper this graph is fully connected (`k_G = 1`), with a density of 13.5%, a median of 4 neighbours per state, ranging from 1 (Baja California Sur) to 9 (San Luis Potosi).

In [ ]:
sg = Graph.build_contiguity(gdf)

In [ ]:
sg.n_components

With `k=2` (the default), `S` splits `PCGDP1940` at its median to form the attribute graph - states above the median are mutually "neighbours" in `A`, as are states below it - and intersects it with the queen contiguity graph. This is the spatial bipolarization index. For 1940, the paper reports a value of 0.87: the low-income states form a single, large connected component running from the southern border up through the centre of the country, while the high-income states are split into five separate spatial clusters.

In [ ]:
s1940 = S(gdf, sg, 'PCGDP1940')

In [ ]:
s1940

To trace how spatial bipolarization evolves, we repeat the calculation for each decade's income variable.

In [ ]:
vars = [f'PCGDP{dec}' for dec in range(1940, 2010, 10)]

In [ ]:
vars

In [ ]:
import numpy
rng = numpy.random.default_rng(42)
res = [S(gdf, sg, var) for var in vars]

In [ ]:
res[0]

In [ ]:
res[1]

In [ ]:
res[2]

In [ ]:
res[3]

In [ ]:
res[4]

In [ ]:
res[5]

In [ ]:
res[6]

## From bipolarization to multipolarization

Splitting at the median is just one choice for the attribute graph. Using `k>2` quantile classes generalizes the bipolarization index to *spatial multipolarization* - the same intersection-graph construction, but now `A` connects locations that fall in the same one of `k` quantile bins rather than just "above" or "below" the median. Here we repeat the calculation for each decade using tertiles (`k=3`).

In [ ]:
res3 = [S(gdf, sg, var,k=3) for var in vars]

In [ ]:
for r in res3:
    print(r)

More generally, we can sweep over several values of `k` (bipolar, tertiles, quintiles, septiles) for every decade to see how sensitive the polarization pattern is to the number of attribute classes. Finer classifications (larger `k`) tend to fragment the attribute graph into more components, which - all else equal - pushes `S` down, so comparisons across `k` are best read as "how does the *relative* ranking across decades change," rather than as directly comparable magnitudes.

In [ ]:
res = [S(gdf, sg, var,k=k) for var in vars for k in [2, 3, 5, 7]]

In [ ]:
for r in res:
    print(r)

It helps to look at the actual quantile classifications behind these numbers before interpreting the index values.

In [ ]:
gdf.plot('PCGDP1940', scheme='quantiles', k=3, legend=True)

In [ ]:
gdf.plot('PCGDP1950', scheme='quantiles', k=3, legend=True)

In [ ]:
gdf.plot('PCGDP1950', scheme='quantiles', k=5, legend=True)

Beyond the summary statistic, an `S` object exposes the full component bookkeeping behind it: the number of components in the intersection, spatial, and attribute graphs, and the per-location labels used to build them.

In [ ]:
res[-1].n_i_components

In [ ]:
res[-1].labels

In [ ]:
r0 = res[-1]

In [ ]:
dir(r0)

In [ ]:
r0.permutations

In [ ]:
r0.statistic_

In [ ]:
r0.p_value

## Comparing polarization across decades and classification schemes

Putting it together, we compute the index, its Monte Carlo p-value, and the number of intersection components for every combination of `k in [2, 3, 5, 7]` and decade, and collect the results into a single dataframe for comparison - in the same spirit as the paper's comparison of the spatial polarization trajectory against Theil's T over time.

In [ ]:
rng = numpy.random.default_rng(42)

res = []
for k in [2, 3, 5, 7]:
    for year in [1940, 1950, 1960, 1970, 1980, 1990, 2000]:
        v = f'PCGDP{year}'
        r = S(gdf, sg, v, k=k)
        res.append([year, k, r.statistic_, r.p_value, r.n_i_components])
        

In [ ]:
import pandas as pd

In [ ]:
res_df = pd.DataFrame(data=np.array(res), 
                      columns=['year', 'k', 's', 'p', 'n_i'])

In [ ]:
res_df.head()

Filtering to the statistically significant results (`p <= 0.05` under the permutation null) highlights which decade/classification combinations show spatial polarization stronger than would be expected if incomes were randomly scattered across the map rather than concentrated by geography.

In [ ]:
res_df[res_df.p<=0.05]

In [ ]:
res_df[res_df.year==2000]

## Summary

Across both the synthetic lattice and the Mexican states application, `S` quantifies something inequality measures like the Gini index or Theil's T cannot: whether similar values are *organized in space*. As Rey (2026) shows for Mexico, national income inequality (Theil's T) and spatial polarization can move independently - inequality between states can fall even while the geographic clustering of rich and poor states persists or intensifies, consistent with a persistent poverty trap in the south alongside a more spatially fragmented set of high-income states elsewhere in the country.